In [ ]:
# =============================================================================
# UMAP: Fit on ChEMBL (train) ONLY, then project COCONUT into the same embedding
#   - Panel A: ChEMBL global UMAP (colored by class)
#   - Panel B: COCONUT projection (all gray + top hits highlighted)
#   - Panel C (optional): COCONUT projection colored by predicted probability
#
# Styling per your request:
#   - larger symbols
#   - small black border
#   - "transparent moss" highlight (moss-green with alpha)
#   - legend order fixed
#
# Requirements (openms_env):
#   pip install numpy pandas scikit-learn matplotlib rdkit-pypi umap-learn
# =============================================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

import umap

from rdkit import Chem
from rdkit.Chem import AllChem, DataStructs, rdMolDescriptors, Descriptors, Crippen
from rdkit.Chem.Scaffolds import MurckoScaffold


# --------------------------- USER PATHS --------------------------------------
BASE = r"C:\Users\Besitzer\Desktop\M3_databases"

TRAIN_CSV = os.path.join(BASE, "ChEMBL_M3_consensus_labels_more_negatives_with_meta.csv")

# Use your ranked COCONUT file if you already have p_antagonist
COCO_CSV  = os.path.join(BASE, "coconut_screen_out", "coconut_screen_ranked.csv")

OUTDIR = os.path.join(BASE, "figures", "UMAP")
os.makedirs(OUTDIR, exist_ok=True)

OUT_A = os.path.join(OUTDIR, "UMAP_train_global")
OUT_B = os.path.join(OUTDIR, "UMAP_coconut_projection_hits")
OUT_C = os.path.join(OUTDIR, "UMAP_coconut_projection_prob")  # optional


# --------------------------- PLOT STYLE --------------------------------------
POINT_SIZE_TRAIN = 24         # larger than before
POINT_SIZE_COCO  = 10         # smaller for huge external library
EDGE_LW = 0.25
EDGE_COLOR = "black"

MOSS = "#7A9E35"              # moss-green-ish
ALPHA_TRAIN = 0.75
ALPHA_BG_COCO = 0.12
ALPHA_HITS = 0.55             # "transparent moss"

RANDOM_STATE = 0

# For COCONUT visualization: to keep plotting fast
COCO_PLOT_SAMPLE_N = 60000    # set None to plot all (not recommended)
TOP_HIT_FRAC = 0.01           # highlight top 1% by p_antagonist (or by model prob if computed here)


# --------------------------- HELPERS -----------------------------------------
def _save_pub_figure(fig, out_prefix, dpi=300):
    png = out_prefix + ".png"
    pdf = out_prefix + ".pdf"
    fig.savefig(png, dpi=dpi, bbox_inches="tight")
    fig.savefig(pdf, bbox_inches="tight")
    print("[SAVED]", png)
    print("[SAVED]", pdf)

def _style_axes(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(axis="both", which="major", labelsize=11)

def get_smiles_col(df, preferred="smiles"):
    if preferred in df.columns:
        return preferred
    for c in ["canonical_smiles", "smiles", "SMILES"]:
        if c in df.columns:
            return c
    raise ValueError("No SMILES column found. Expected one of canonical_smiles/smiles/SMILES.")

def smiles_to_mol(s):
    if pd.isna(s):
        return None
    m = Chem.MolFromSmiles(str(s))
    return m

def morgan_fp(mol, radius=2, n_bits=2048, use_chirality=True, use_features=False):
    if mol is None:
        return None
    return AllChem.GetMorganFingerprintAsBitVect(
        mol, radius, nBits=n_bits, useChirality=use_chirality, useFeatures=use_features
    )

def fps_to_numpy(bitvect_list):
    """Convert RDKit ExplicitBitVect list -> numpy uint8 (n, n_bits)."""
    n = len(bitvect_list)
    n_bits = bitvect_list[0].GetNumBits()
    X = np.zeros((n, n_bits), dtype=np.uint8)
    for i, bv in enumerate(bitvect_list):
        arr = np.zeros((n_bits,), dtype=np.int8)
        DataStructs.ConvertToNumpyArray(bv, arr)
        X[i, :] = arr.astype(np.uint8)
    return X

def compute_physchem(mol):
    # MW (Da), logP (dimensionless), TPSA (Å^2), HBD/HBA (counts), RotB (count)
    return [
        Descriptors.MolWt(mol),
        Crippen.MolLogP(mol),
        rdMolDescriptors.CalcTPSA(mol),
        rdMolDescriptors.CalcNumHBD(mol),
        rdMolDescriptors.CalcNumHBA(mol),
        rdMolDescriptors.CalcNumRotatableBonds(mol),
    ]

def build_X_fp_only(df, smiles_col, n_bits=2048):
    mols = [smiles_to_mol(s) for s in df[smiles_col].astype(str)]
    fps = [morgan_fp(m, radius=2, n_bits=n_bits, use_chirality=True, use_features=False) for m in mols]
    ok = [fp is not None for fp in fps]
    df2 = df.loc[ok].copy()
    fps = [fp for fp in fps if fp is not None]
    X = fps_to_numpy(fps).astype(np.float32)
    return df2, X

def build_X_fp_phys(df, smiles_col, n_bits=2048):
    mols = [smiles_to_mol(s) for s in df[smiles_col].astype(str)]
    fps = [morgan_fp(m, radius=2, n_bits=n_bits, use_chirality=True, use_features=False) for m in mols]
    ok = [m is not None and fp is not None for m, fp in zip(mols, fps)]
    df2 = df.loc[ok].copy()
    mols2 = [m for m, keep in zip(mols, ok) if keep]
    fps2 = [fp for fp, keep in zip(fps, ok) if keep]
    Xfp = fps_to_numpy(fps2).astype(np.float32)
    Xpc = np.array([compute_physchem(m) for m in mols2], dtype=np.float32)
    X = np.hstack([Xfp, Xpc])
    return df2, X

def make_train_pipeline(fp_plus_physchem=False):
    # NOTE: StandardScaler(with_mean=False) is fine for sparse-like FP matrices
    # For FP+physchem, it's still OK (physchem columns are dense but not harmed).
    return Pipeline([
        ("scaler", StandardScaler(with_mean=False)),
        ("clf", LogisticRegression(
            solver="liblinear",
            max_iter=5000,
            class_weight="balanced",
            penalty="l2",
            C=1.0,
        ))
    ])

def maybe_sample(df, n, seed=0):
    if n is None or len(df) <= n:
        return df
    return df.sample(n=n, random_state=seed)


# =============================================================================
# 1) LOAD TRAIN + BUILD TRAIN EMBEDDING (fit UMAP on TRAIN ONLY)
# =============================================================================
train = pd.read_csv(TRAIN_CSV)
train_smiles = get_smiles_col(train, preferred="smiles")

POS = {"active", "active_single"}
NEG = {"inactive", "inactive_single"}

train = train[train["consensus_label"].isin(POS | NEG)].copy()
train["y"] = train["consensus_label"].isin(POS).astype(int)

# weights (keep consistent with your modeling)
w_map = {"active": 1.0, "inactive": 1.0, "active_single": 0.5, "inactive_single": 0.7}
train["w"] = train["consensus_label"].map(w_map).astype(float)

print("[TRAIN] loaded:", train.shape, "| pos_frac:", train["y"].mean())

# Choose which representation drives UMAP:
#   - For manuscript clarity, use FP-only UMAP (most common)
FP_PLUS_PHYSCHEM_FOR_UMAP = False

if FP_PLUS_PHYSCHEM_FOR_UMAP:
    train2, X_train = build_X_fp_phys(train, train_smiles, n_bits=2048)
else:
    train2, X_train = build_X_fp_only(train, train_smiles, n_bits=2048)

y_train = train2["y"].values.astype(int)

print("[TRAIN] usable:", train2.shape, "| X_train:", X_train.shape)

# Fit UMAP on TRAIN ONLY
reducer = umap.UMAP(
    n_neighbors=15,
    min_dist=0.10,
    n_components=2,
    metric="jaccard" if not FP_PLUS_PHYSCHEM_FOR_UMAP else "euclidean",
    random_state=RANDOM_STATE,
    transform_seed=RANDOM_STATE,
)
emb_train = reducer.fit_transform(X_train)

# =============================================================================
# 2) PANEL A — TRAIN GLOBAL UMAP (class colors)
# =============================================================================
fig, ax = plt.subplots(figsize=(7.2, 6.2))

# Inactive first, Active second (legend order)
mask0 = (y_train == 0)
mask1 = (y_train == 1)

ax.scatter(
    emb_train[mask0, 0], emb_train[mask0, 1],
    s=POINT_SIZE_TRAIN, alpha=ALPHA_TRAIN,
    edgecolors=EDGE_COLOR, linewidths=EDGE_LW,
    label="Inactive (0)"
)
ax.scatter(
    emb_train[mask1, 0], emb_train[mask1, 1],
    s=POINT_SIZE_TRAIN, alpha=ALPHA_TRAIN,
    edgecolors=EDGE_COLOR, linewidths=EDGE_LW,
    label="Active (1)"
)

ax.set_title("UMAP (fit on ChEMBL training set) — Morgan fingerprints", fontsize=13)
ax.set_xlabel("UMAP-1")
ax.set_ylabel("UMAP-2")
_style_axes(ax)
ax.legend(frameon=False, loc="best")
fig.tight_layout()
_save_pub_figure(fig, OUT_A)
plt.close(fig)


# =============================================================================
# 3) LOAD COCONUT + PROJECT INTO THE TRAIN UMAP SPACE
# =============================================================================
coco = pd.read_csv(COCO_CSV)
coco_smiles = get_smiles_col(coco, preferred="smiles")

# if p_antagonist already exists, we use it; otherwise we compute using a fitted model
HAS_SCORE = ("p_antagonist" in coco.columns)

# Build X_coco in the SAME representation used for UMAP
if FP_PLUS_PHYSCHEM_FOR_UMAP:
    coco2, X_coco = build_X_fp_phys(coco, coco_smiles, n_bits=2048)
else:
    coco2, X_coco = build_X_fp_only(coco, coco_smiles, n_bits=2048)

print("[COCO] usable:", coco2.shape, "| X_coco:", X_coco.shape)

# Project COCONUT into training-defined embedding
emb_coco = reducer.transform(X_coco)

# Determine score for ranking/highlights
if HAS_SCORE:
    p = coco2["p_antagonist"].values.astype(float)
else:
    # Compute with a model trained on ALL ChEMBL (same feature type as scoring)
    # (This is for plotting convenience; it matches your pipeline idea.)
    pipe = make_train_pipeline(fp_plus_physchem=False)  # scoring model is typically FP-only or FP+physchem
    # For scoring we use FP-only to be consistent with your baseline unless you prefer otherwise.
    train_scoring, X_scoring = build_X_fp_only(train, train_smiles, n_bits=2048)
    y_scoring = train_scoring["consensus_label"].isin(POS).astype(int).values
    w_scoring = train_scoring["consensus_label"].map(w_map).astype(float).values

    pipe.fit(X_scoring, y_scoring, clf__sample_weight=w_scoring)

    # IMPORTANT: scoring features must match pipe input => FP-only.
    coco_scoring, X_coco_scoring = build_X_fp_only(coco2, coco_smiles, n_bits=2048)
    # align arrays after possible drop:
    # simplest: recompute p on coco2 by index intersection
    p_tmp = pipe.predict_proba(X_coco_scoring)[:, 1]
    coco2 = coco_scoring.copy()
    X_coco = X_coco_scoring
    emb_coco = reducer.transform(X_coco)
    p = p_tmp

# Sample for plotting (fast)
plot_idx = np.arange(len(coco2))
if COCO_PLOT_SAMPLE_N is not None and len(coco2) > COCO_PLOT_SAMPLE_N:
    rng = np.random.default_rng(RANDOM_STATE)
    plot_idx = rng.choice(plot_idx, size=COCO_PLOT_SAMPLE_N, replace=False)

# Top-hit selection (on full coco2, not just sample)
cut = np.quantile(p, 1.0 - TOP_HIT_FRAC)
is_hit = (p >= cut)

# =============================================================================
# 4) PANEL B — COCONUT projection (background + moss hits)
# =============================================================================
fig, ax = plt.subplots(figsize=(7.2, 6.2))

# Background (sampled)
ax.scatter(
    emb_coco[plot_idx, 0], emb_coco[plot_idx, 1],
    s=POINT_SIZE_COCO, alpha=ALPHA_BG_COCO,
    edgecolors="none",
    label="COCONUT (background)"
)

# Hits (plot all hits; if too many, it still stays manageable at TOP_HIT_FRAC)
hit_idx = np.where(is_hit)[0]
ax.scatter(
    emb_coco[hit_idx, 0], emb_coco[hit_idx, 1],
    s=max(POINT_SIZE_COCO, 14),
    alpha=ALPHA_HITS,
    c=MOSS,
    edgecolors=EDGE_COLOR, linewidths=EDGE_LW,
    label=f"Top {int(TOP_HIT_FRAC*100)}% by score"
)

ax.set_title("COCONUT projected into ChEMBL UMAP — top hits highlighted", fontsize=13)
ax.set_xlabel("UMAP-1")
ax.set_ylabel("UMAP-2")
_style_axes(ax)

# Legend order fixed explicitly:
handles, labels = ax.get_legend_handles_labels()
order = [labels.index("COCONUT (background)"), labels.index(f"Top {int(TOP_HIT_FRAC*100)}% by score")]
ax.legend([handles[i] for i in order], [labels[i] for i in order], frameon=False, loc="best")

fig.tight_layout()
_save_pub_figure(fig, OUT_B)
plt.close(fig)

# =============================================================================
# 5) PANEL C (OPTIONAL) — COCONUT projection colored by probability
# =============================================================================
fig, ax = plt.subplots(figsize=(7.2, 6.2))

sc = ax.scatter(
    emb_coco[plot_idx, 0], emb_coco[plot_idx, 1],
    s=POINT_SIZE_COCO, alpha=0.80,
    c=p[plot_idx],
    edgecolors=EDGE_COLOR, linewidths=EDGE_LW,
)
cb = fig.colorbar(sc, ax=ax)
cb.set_label("Predicted antagonist score (p_antagonist)", rotation=90)

ax.set_title("COCONUT projected into ChEMBL UMAP — colored by predicted score", fontsize=13)
ax.set_xlabel("UMAP-1")
ax.set_ylabel("UMAP-2")
_style_axes(ax)

fig.tight_layout()
_save_pub_figure(fig, OUT_C)
plt.close(fig)

print("\nDone. Wrote:")
print(" -", OUT_A + ".pdf/.png")
print(" -", OUT_B + ".pdf/.png")
print(" -", OUT_C + ".pdf/.png")
